# Entrega 2 proyecto aprendizaje de máquina

Pedro Pablo Sanín Trujillo (202221527)


## 1 - Carga de datos


In [2]:
# Importar la librerías del proyecto
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

In [3]:
# A continuación, se cargan los conjuntos de datos de entrenamiento y evaluación

path_train = "./datos/train.csv"
path_eval = "./datos/eval.csv"


dataset_train_original = pd.read_csv(path_train)
dataset_eval_original = pd.read_csv(path_eval)

dataset_train = dataset_train_original.copy()
dataset_eval = dataset_eval_original.copy()

In [4]:
print(
    f"Número de filas y columnas del conjunto de entrenamiento: {dataset_train.shape}"
)

dataset_train.head(10)

Número de filas y columnas del conjunto de entrenamiento: (20466, 3)


,id,text,label
0,0,Lo pedí un martes por internet y llegó al terc...,negativo
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo
2,2,Lo compré hace como dos meses en una tienda de...,neutral
3,3,Le routeur capte très bien dans toute la maiso...,negativo
4,4,"I got it a while back on sale online, it arriv...",neutral
5,5,Comprei esse teclado mês passado pra trabalhar...,positivo
6,6,Comprei há umas duas semanas pela internet e c...,negativo
7,7,"La compre a mediados de año por internet, tard...",positivo
8,8,Les dejo un comentario rápido. Lo pedí a media...,negativo
9,9,The color accuracy on this monitor is genuinel...,positivo


Vemos que el dataset contiene 20466 filas, cada una con un id, la reseña correspondiente y un label que indica que si la reseña es positiva o negativa. Revisando manualmente algunas entradas, confirmamos que los únicos idiomas presentes en las reseñas son español, inglés, portugués, francés y una mezcla de español e inglés. Esto es consistente con lo indicado en el enunciado del proyecto. Preliminarmente, no separaremos los modelos reseñas por idioma pues esto reduce sustancialmente el tamaño de cada conjunto de datos. Sin embargo, si vale la pena identificar el idioma para tener una división train-test balanceada.

Además nuestra métrica de entrenamiento será el accuracy, de acuerdo con la competencia. A continuación, intentaremos identificar el idioma de cada una de nuestras reseñas.


## 2 - Identificando el idioma de la reseña

Para esto, descargaremos las 50 palabras más comunes en cada uno de los idiomas. Por la ley de Zipf, si las listas de las 50 palabras más comunes son distintas, una de las palabras aparecerá en la lista y esto nos servirá para determinar el idioma de la reseña. Asimismo, las palabras más comunes en cada idioma son artículos y generalmente tienen el mismo significado.

Así, el criterio para determinar el idioma de una reseña será el siguiente:

1. El primer y segundo idioma candidato serán los primeros dos conjuntos de palabras que tengan más coincidencia con las palabras de la reseña.

2. Con las palabras que pertenecen al conjunto más común,


In [5]:
import sys
import unicodedata
from collections import Counter

# Tabla de traducción que reemplaza por un espacio todo carácter Unicode de puntuación
# (categorías 'P*'), lo que incluye ¿ ¡ « » “ ” … además de la puntuación ASCII.
puntua = {
    i: " "
    for i in range(sys.maxunicode + 1)
    if unicodedata.category(chr(i)).startswith("P")
}

# Palabras de la lista de inglés que también son muy comunes en español ("lo he usado"),
# por lo que no cuentan como evidencia de inglés dentro de una reseña en español.
AMBIGUAS_EN = {"he"}


def procesar_resena(text):
    """Pasa la reseña a minúsculas, elimina la puntuación y la separa en palabras."""
    text = text.lower()
    text = text.translate(puntua)
    return text.split()


def palabras_exclusivas(palabras_por_idioma):
    """Para cada idioma, las palabras comunes que no aparecen en la lista de ningún otro idioma."""
    return {
        idioma: palabras
        - set().union(
            *(otras for otro, otras in palabras_por_idioma.items() if otro != idioma)
        )
        for idioma, palabras in palabras_por_idioma.items()
    }


def identificar_idioma(resena, palabras_por_idioma, exclusivas):
    """Identifica el idioma de una reseña: 'es', 'en', 'fr', 'pt_br', 'es_en' (bilingüe) o None.

    1. El idioma principal es el que tiene más ocurrencias de sus palabras exclusivas. Si la reseña
       no tiene ninguna palabra exclusiva, se usan todas las palabras comunes (incluidas las compartidas).
    2. Si el idioma principal es español o inglés y la reseña contiene al menos una palabra exclusiva
       del otro idioma, la reseña se considera bilingüe ('es_en').
    """
    conteo = Counter(procesar_resena(resena))

    def puntajes(listas):
        return {
            idioma: sum(n for palabra, n in conteo.items() if palabra in palabras)
            for idioma, palabras in listas.items()
        }

    puntaje = puntajes(exclusivas)
    if max(puntaje.values()) == 0:
        puntaje = puntajes(palabras_por_idioma)
        if max(puntaje.values()) == 0:
            return None
    principal = max(puntaje, key=lambda k: puntaje[k])

    hay_es = any(palabra in exclusivas["es"] for palabra in conteo)
    hay_en = any(palabra in exclusivas["en"] - AMBIGUAS_EN for palabra in conteo)
    if principal in ("es", "en") and hay_es and hay_en:
        return "es_en"
    return principal

In [6]:
# Cargamos las 50 palabras más comunes por idioma y les aplicamos la misma limpieza que a las reseñas,
# para que entradas como "c'", "l'" o "'s" queden como "c", "l" y "s" y puedan coincidir con las reseñas.
path_palabras = "./datos/50_palabras_mas_comunes/50_palabras_mas_comunes.csv"

palabras_comunes = pd.read_csv(path_palabras)
palabras_comunes["palabra"] = palabras_comunes["palabra"].map(
    lambda p: procesar_resena(p)[0]
)

# Conjunto de palabras por idioma, y el subconjunto de palabras exclusivas de cada idioma
palabras_por_idioma = palabras_comunes.groupby("idioma")["palabra"].apply(set).to_dict()
exclusivas = palabras_exclusivas(palabras_por_idioma)

palabras_comunes.head()

,idioma,rank,palabra,frecuencia
0,en,1,you,28787591
1,en,2,i,27086011
2,en,3,the,22761659
3,en,4,to,17099834
4,en,5,a,14484562


In [7]:
print(
    identificar_idioma("Esta es una reseña de prueba", palabras_por_idioma, exclusivas)
)
print(identificar_idioma("This is a test review", palabras_por_idioma, exclusivas))
print(
    identificar_idioma("Ceci est une critique de test", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Esta é uma crítica de teste", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en inglés: This is a test review",
        palabras_por_idioma,
        exclusivas,
    )
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en francés: Ceci est une critique de test",
        palabras_por_idioma,
        exclusivas,
    )
)

es
en
fr
pt_br
es_en
es


In [8]:
# Identificamos el idioma de cada reseña de entrenamiento y evaluación
dataset_train["idioma"] = dataset_train["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)
dataset_eval["idioma"] = dataset_eval["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)

dataset_train["idioma"].value_counts(dropna=False, normalize=True)

idioma
es       0.652497
en       0.114678
pt_br    0.114043
fr       0.102511
es_en    0.016271
Name: proportion, dtype: float64

In [9]:
dataset_train.head()

,id,text,label,idioma
0,0,Lo pedí un martes por internet y llegó al terc...,negativo,es
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo,es
2,2,Lo compré hace como dos meses en una tienda de...,neutral,es
3,3,Le routeur capte très bien dans toute la maiso...,negativo,fr
4,4,"I got it a while back on sale online, it arriv...",neutral,en


Ahora, dividimos el conjunto de datos en prueba y entrenamiento estratificado para incluir una proporción representativa de cada idioma y de cada etiqueta. Para esto, usaremos `sklearn.train_test_split`.


In [10]:
# Codificamos las etiquetas de como enteros
etiquetas = sorted(
    dataset_train["label"].unique()
)  # ['negativo', 'neutral', 'positivo']
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}


# División 90/10 estratificada por idioma
train_df, test_df = train_test_split(
    dataset_train,
    test_size=0.1,
    random_state=42,
    stratify=dataset_train["idioma"],
)


def dataset_a_tf(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["text"].values, df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

## 2 - Red neuronal

In [11]:
encoder_mlp = TextVectorization(output_mode="tf_idf")
encoder_mlp.adapt(train_df["text"])

In [12]:
model_mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_mlp,
    
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model_mlp.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 19025)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       608,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 608,931 (2.32 MB)

 Trainable params: 608,931 (2.32 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model_mlp.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

Ahora para entrenar el modelo, usaremos ``tf.keras.callbacks.EarlyStopping``. Esto nos permite parar el entrenamiento cuando maximicemos la métrica objetivo, y restaurar los mejores parámetros en caso de que la métrica objetivo baje luego de la iteración.

In [26]:
# En lugar de fijar el número de épocas, entrenamos hasta un máximo de 50 y detenemos el
# entrenamiento cuando el accuracy de validación deja de mejorar durante 3 épocas seguidas.
# restore_best_weights=True deja el modelo con los pesos de la mejor época.
def entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset, max_epochs=50, patience=3): 
    parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=patience,
    restore_best_weights=True,
)

    history = model.fit(
    train_dataset,
    epochs=max_epochs,
    validation_data=test_dataset,
    callbacks=[parada_temprana],
)

    mejor_epoca = max(
    range(len(history.history["val_accuracy"])),
    key=lambda i: history.history["val_accuracy"][i],
)
    print(
    f"Mejor época: {mejor_epoca + 1} "
    f"(val_accuracy = {history.history['val_accuracy'][mejor_epoca]:.4f})"
)



In [25]:
entrenar_modelo_con_parada_temprana(model_mlp, train_dataset, test_dataset)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9935 - loss: 0.0228 - val_accuracy: 0.6844 - val_loss: 1.4483
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9932 - loss: 0.0250 - val_accuracy: 0.7015 - val_loss: 1.4959
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9947 - loss: 0.0189 - val_accuracy: 0.6825 - val_loss: 1.6234
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9963 - loss: 0.0140 - val_accuracy: 0.6898 - val_loss: 1.6401
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9974 - loss: 0.0090 - val_accuracy: 0.6908 - val_loss: 1.6664
Mejor época: 2 (val_accuracy = 0.7015)


### 3.1 - Búsqueda de hiperparámetros 

## 3 - Red neuronal recurrente básica


Inicialmente, usaremos una red neuronal recurrente para clasificar las reseñas. Inicialmente, veremos el desempeño de una RNN simple, donde cada neurona tiene información sobre las operaciones que ha hecho en pasos anteriores. Las redes neuronales recurrentes funcionan bien al procesar lenguaje natural pues el lenguaje tiene una naturaleza secuencial.

Usaremos la implementación de RNNs dada por `tf.keras.layers.SimpleRNN`. Para que el texto pueda pasar exitosamente por una red neuronal, debemos vectorizarlo. Para esto, usaremos una capa inicial de la red neuronal dada por `tf.keras.layers.TextVectorization`. Este algoritmo de vectorización combina varias formas de vectorizar frases como _tf_idf_, _count_ (bag of words) y soporta ngramas.


In [20]:
# Para una red recurrente necesitamos una secuencia de índices de palabras (output_mode='int'),
# no un vector tf_idf. Cada reseña se rellena o recorta a 100 palabras (mediana: 57, máximo: 125).
encoder_rnn = TextVectorization(output_mode="int", output_sequence_length=100)
encoder_rnn.adapt(train_df["text"])

texto_original = train_df["text"][1]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))
print(f"Texto original: {texto_original}")
print(f"Texto vectorizado: {texto_vectorizado}")

Texto original: Les cuento como me fue. Compre esta tablet a mediados de año y llego en cuatro dias a mi casa en Guadalajara, sin ningun problema con el paquete. En mi opinion, la resolucion me resulto ser deficiente para ser honesto. La uso sobre todo por las tardes para revisar correos y ver videos mientras ceno. La pantalla tactil termino siendo muy buena en el uso real.
Texto vectorizado: [[  80  711   73   10  184   61  111 1855    7   69    3  158    9   43
     4   50   30    7   19   23    4  409   40 1683  154   16    5   94
     4   19 1032    2 1289   10  201  110 1017   12  110  436    2   20
   151   51   13   35  166   12  741 1716    9  481 1053  373 8252    2
   263 1468  516  212  172  492    4    5   20  379    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]]


Ahora, entrenaremos una red neuronal con una capa de embedding, una capa SimpleRNN de 64 neuronas y una capa oculta densa de 32 neuronas con función de activación ReLU. Asimismo, entrenamos un embedding con  


In [21]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_rnn,
        tf.keras.layers.Embedding(
            input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
        ),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 64)        │     1,217,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,228,099 (4.68 MB)

 Trainable params: 1,228,099 (4.68 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [28]:
entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9828 - loss: 0.0464 - val_accuracy: 0.7294 - val_loss: 1.0260
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9867 - loss: 0.0376 - val_accuracy: 0.7313 - val_loss: 1.0514
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9895 - loss: 0.0301 - val_accuracy: 0.7377 - val_loss: 1.0916
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9855 - loss: 0.0387 - val_accuracy: 0.7289 - val_loss: 1.1346
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9914 - loss: 0.0242 - val_accuracy: 0.7411 - val_loss: 1.3348
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9931 - loss: 0.0193 - val_accuracy: 0.7445 - val_loss: 1.2558
Epoch 7/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9922 - loss: 0.0226 - val_accuracy: 0.7386 - val_loss: 1.2707
Epoch 8/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9866 - loss: 0.0400 - val_accu